In [ ]:
%pip install requests beautifulsoup4 google-colab

In [ ]:
import requests
from bs4 import BeautifulSoup
from google.colab import drive
import pandas as pd

In [ ]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
base_url = "https://remax-central.com.sv/es"
page_number = 1
url = f"{base_url}?page={page_number}"
print(url)

https://remax-central.com.sv/es?page=1


In [ ]:
response = requests.get(url)
response.text

'<!DOCTYPE html>\n<html lang="es">\n  <head>\n    <meta charset="utf-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1, maximum-scale=1">\n    <title>RE/MAX Central El Salvador</title>\n<meta name="description" content="Bienes Raíces El Salvador">\n<meta name="keywords" content="bienes raices el salvador, bienes raíces, alquiler de casas el salvador, venta de casas el salvador, renta de casas el salvador, compra de casas el salvador">\n<link rel="canonical" href="https://remax-central.com.sv/es"/>\n<meta property="og:title" content="RE/MAX Central El Salvador" />\n<meta property="og:description" content="Bienes Raíces El Salvador" />\n<meta property="og:url" content="https://remax-central.com.sv/es" />\n<meta property="og:site_name" content="RE/MAX Central El Salvador" />\n\n\n    <link rel="apple-touch-icon-precomposed" sizes="57x57" href="https://remax-central.com.sv/images/icon/apple-touch-icon-57x57.png" />\n<link rel="apple-touch-icon-precomposed" sizes="

In [18]:
soup = BeautifulSoup(response.content, 'html.parser')

total_pages = 1

pagination_nav = soup.find('ul', class_='pagination')
page_links = pagination_nav.find_all('a', href=True)

page_numbers = []
for link in page_links:
    href = link['href']
    if 'page=' in href:
        try:
            page_number = int(href.split('page=')[1])
            page_numbers.append(page_number)
        except ValueError:
            continue

total_pages = max(page_numbers) if page_numbers else 1
print(f"Total pages: {total_pages}")

Total pages: 22


In [30]:
import requests
from bs4 import BeautifulSoup
import re

def extract_page_structure(page_num=1):
    url = f"https://remax-central.com.sv/es?page={page_num}"
    resp = requests.get(url)
    resp.encoding = resp.apparent_encoding
    soup = BeautifulSoup(resp.text, "html.parser")

    # Intentamos identificar contenedores repetidos
    # Ajusta el selector si identificas una clase específica
    items = soup.find_all(lambda tag: tag.name and tag.find(text=re.compile(r"\$")))

    results = []
    for item in items:
        # Buscar ancestros que agrupen cada bloque completo
        container = item.find_parent()
        title = container.find(text=re.compile(r'\w')).strip() if container else None

        # Ubicación (lugar geográfico)
        location = None
        nexts = container.find_all_next(text=True, limit=5)
        for t in nexts:
            if t.strip() in ["Venta", "Alquiler"]:
                break
            if re.match(r'^[A-Z][a-z]', t):
                location = t.strip()
                break

        price_match = re.search(r"\$\d[\d,]*", container.text)
        price = price_match.group(0) if price_match else None

        tipo = None
        if "Venta" in container.text:
            tipo = "Venta"
        elif "Alquiler" in container.text:
            tipo = "Alquiler"

        # Buscar patrones xN junto a imágenes o texto
        rooms = re.search(r"Rooms\D*x(\d+)", container.text) or re.search(r"x(\d+)\s*\n", container.text)
        rooms = rooms.group(1) if rooms else None

        levels = re.search(r"Levels\D*x(\d+)", container.text) or re.search(r"x(\d+)\s*\n", container.text)
        levels = levels.group(1) if levels else None

        parking = re.search(r"Parking\D*x(\d+)", container.text)
        parking = parking.group(1) if parking else None

        results.append({
            "title": title,
            "location": location,
            "price": price,
            "type": tipo,
            "rooms": rooms,
            "levels": levels,
            "parking": parking
        })

    return results

data = extract_page_structure(1)
for d in data:
    print(d)
